<!-- Cell 1 -->
# 01 — Aggregate Per-Class Datasets to 200/class

Pools every (image, label) pair across all sub-datasets/splits per class, samples 200, writes to `data/aggregated/<class>/` with naming `<class>_<NNNN>.jpg` (0001–0200). Labels keep `class_id = 0`; the global remap happens in notebook 02.

In [10]:
# Cell 2
# Install dependencies
%pip install -q pandas pillow pyyaml
print("nb01 dependencies ready")

Note: you may need to restart the kernel to use updated packages.
nb01 dependencies ready



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [11]:
# Cell 3
## Imports
import json, random
from pathlib import Path
import pandas as pd
import yaml
from PIL import Image, ImageOps

In [12]:
# Cell 4
# Config — paths, classes, sampling
SRC = Path('../datasets')         # raw per-class sub-dataset exports (Roboflow)
DST = Path('../data/aggregated')  # output: 200 jpgs + 200 labels per class

CLASSES = ['projector', 'whiteboard', 'fire_extinguisher', 'door_sign']
TARGET_PER_CLASS = 200
SEED = 42

# Source-export filesystem conventions
IMG_EXTS = {'.jpg', '.jpeg', '.png', '.webp'}
SPLIT_DIRS = {'train', 'valid', 'val', 'test'}  # Roboflow uses 'valid'
JPEG_QUALITY = 92

random.seed(SEED)
assert SRC.exists(), f'create per-class folders under {SRC.resolve()} and drop exports into them'
for c in CLASSES:
    (DST / c / 'images').mkdir(parents=True, exist_ok=True)
    (DST / c / 'labels').mkdir(parents=True, exist_ok=True)

<!-- Cell 5 -->
## 1. Discover (image, label) pairs
Walk every sub-dataset and split under `datasets/<class>/`, recording source-dataset and split for provenance.

In [13]:
# Cell 6
# Collect (image, label) pairs per class; print available counts
def find_label(img_path: Path) -> Path | None:
    for cand in [img_path.parent.parent / 'labels' / (img_path.stem + '.txt'),
                 img_path.with_suffix('.txt')]:
        if cand.exists():
            return cand
    return None

def infer_split(img_path: Path) -> str:
    for part in img_path.parts[::-1]:
        if part in SPLIT_DIRS:
            return 'val' if part == 'valid' else part
    return 'train'

pairs_per_class: dict[str, list[dict]] = {}
for c in CLASSES:
    class_root = SRC / c
    if not class_root.exists():
        print(f'[missing] {class_root}'); pairs_per_class[c] = []; continue
    collected = []
    for sub in sorted(p for p in class_root.iterdir() if p.is_dir()):
        for img in sub.rglob('*'):
            if img.suffix.lower() not in IMG_EXTS:
                continue
            lbl = find_label(img)
            if lbl is None:
                continue
            collected.append({'img': img, 'lbl': lbl,
                              'source_dataset': sub.name, 'source_split': infer_split(img)})
    pairs_per_class[c] = collected

pd.DataFrame([(c, len(p)) for c, p in pairs_per_class.items()],
             columns=['class', 'available_pairs'])

,class,available_pairs
0,projector,249
1,whiteboard,238
2,fire_extinguisher,248
3,door_sign,240


<!-- Cell 7 -->
## 2. Source breakdown
Pairs grouped by `(class, source_dataset, source_split)` — confirms each class draws from the expected exports.

In [14]:
# Cell 8
# Pairs per class × source_dataset × split
rows = [{'class': c, **p} for c, pairs in pairs_per_class.items() for p in pairs]
breakdown = pd.DataFrame(rows).drop(columns=['img', 'lbl'], errors='ignore')
breakdown.groupby(['class', 'source_dataset', 'source_split']).size().unstack(fill_value=0)

source_split                      train
class             source_dataset       
door_sign         doorsign1          56
                  doorsign2          40
                  doorsign3          59
                  doorsign4          85
fire_extinguisher train             248
projector         train             249
whiteboard        train             238

<!-- Cell 9 -->
## 3. Sample 200/class → write to `data/aggregated/`
Shuffle each class pool, take `TARGET_PER_CLASS`, re-encode as RGB JPEG, rename `<class>_<NNNN>.jpg`. Labels keep `class_id = 0`. Provenance recorded in `info.json`.

In [15]:
# Cell 10
# Sample, copy/re-encode, write per-class info.json
for c in CLASSES:
    pool = pairs_per_class[c]
    if len(pool) < TARGET_PER_CLASS:
        print(f'[warn] {c}: only {len(pool)} pairs available (< {TARGET_PER_CLASS})')
    random.shuffle(pool)
    selected = pool[:TARGET_PER_CLASS]

    info = {'class': c, 'count': len(selected), 'items': []}
    for i, p in enumerate(selected, 1):
        stem = f'{c}_{i:04d}'
        dst_img = DST / c / 'images' / f'{stem}.jpg'
        dst_lbl = DST / c / 'labels' / f'{stem}.txt'
        with Image.open(p['img']) as im:
            ImageOps.exif_transpose(im).convert('RGB').save(dst_img, 'JPEG', quality=JPEG_QUALITY)
        dst_lbl.write_text(p['lbl'].read_text())
        info['items'].append({
            'filename': f'{stem}.jpg',
            'source_dataset': p['source_dataset'],
            'source_split': p['source_split'],
            'original_image': str(p['img']),
        })

    (DST / c / 'info.json').write_text(json.dumps(info, indent=2))
    print(f'{c}: wrote {len(selected)} pairs to {DST / c}')

projector: wrote 200 pairs to ..\data\aggregated\projector
whiteboard: wrote 200 pairs to ..\data\aggregated\whiteboard
fire_extinguisher: wrote 200 pairs to ..\data\aggregated\fire_extinguisher
door_sign: wrote 200 pairs to ..\data\aggregated\door_sign


<!-- Cell 11 -->
## 4. Verify
Each class should show 200 images, 200 labels, and `info.json: True`.

In [16]:
# Cell 12
# Count outputs per class
check = []
for c in CLASSES:
    imgs = list((DST / c / 'images').glob('*.jpg'))
    lbls = list((DST / c / 'labels').glob('*.txt'))
    info_ok = (DST / c / 'info.json').exists()
    check.append({'class': c, 'images': len(imgs), 'labels': len(lbls), 'info.json': info_ok})
pd.DataFrame(check)

,class,images,labels,info.json
0,projector,200,200,True
1,whiteboard,200,200,True
2,fire_extinguisher,200,200,True
3,door_sign,200,200,True


<!-- Cell 13 -->
**Done.** All four classes have 200 pairs + `info.json`. Continue to **notebook 02** to merge them into a single split dataset.